# Mineria de datos - clase 9
## alumno: Enzo Ariel Melian

### 1. Preparacion de los datos

In [7]:
import pandas as pd
import numpy as np
import nltk
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Descargar recursos necesarios de NLTK
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

# Datos de ejemplo (opiniones de clientes)
opiniones = [ "Este producto es excelente, me encantó",
                                     "Horrible, nunca lo compraría",
                                      "Muy bueno, lo recomiendo totalmente",
                                      "No me gustó, la calidad es mala",
                                      "Una compra increíble, vale la pena",
                                      "Muy malo, pésima experiencia",
                                      ]

# Etiquetas: 1 = Positivo, 0 = Negativo
etiquetas = np.array([1, 0, 1, 0, 1, 0])

# Función de preprocesamiento del texto
def limpiar_texto(texto):
  texto = texto.lower().translate(str.maketrans("", "", string.punctuation))
  tokens = word_tokenize(texto)
  tokens_limpios = [word for word in tokens if word not in stopwords.words("spanish")]
  return " ".join(tokens_limpios)

#Aplicar preprocesamiento
opiniones_limpias = [limpiar_texto(op) for op in opiniones]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### 2. Clasificacion con Scikit-learn

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Dividir en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(opiniones_limpias, etiquetas, test_size=0.2, random_state=42)

# Vectorización TF-IDF + Naive Bayes
modelo_nb = make_pipeline(TfidfVectorizer(), MultinomialNB())
modelo_nb.fit(X_train, y_train)

# Evaluar modelo
y_pred_nb = modelo_nb.predict(X_test)
print("Precisión Naive Bayes:", accuracy_score(y_test, y_pred_nb))
print("\nReporte de Clasificación:\n", classification_report(y_test, y_pred_nb))

Precisión Naive Bayes: 0.5

Reporte de Clasificación:
               precision    recall  f1-score   support

           0       0.50      1.00      0.67         1
           1       0.00      0.00      0.00         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### 3. Clasificacion con Tensorflow

In [9]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Embedding, Dense, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Tokenización del texto con TensorFlow
tokenizer = Tokenizer(num_words=1000, oov_token="<OOV>")
tokenizer.fit_on_texts(opiniones_limpias)
sequences = tokenizer.texts_to_sequences(opiniones_limpias)
padded_sequences = pad_sequences(sequences, maxlen=10, padding='post')

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(padded_sequences, etiquetas, test_size=0.2, random_state=42)

# Definir el modelo de red neuronal
modelo_tf = keras.Sequential([ Embedding(input_dim=1000,
                                         output_dim=16,
                                         input_length=10),
                               LSTM(16),
                               Dense(8, activation='relu'),
                               Dense(1, activation='sigmoid')
                               ])

# Compilar y entrenar el modelo
modelo_tf.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
modelo_tf.fit(X_train, y_train, epochs=5, verbose=1)

# Evaluar modelo
y_pred_tf = (modelo_tf.predict(X_test) > 0.5).astype("int32")
print("Precisión Red Neuronal:", accuracy_score(y_test, y_pred_tf))

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.5000 - loss: 0.6928
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.7500 - loss: 0.6927
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.5000 - loss: 0.6925
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.7500 - loss: 0.6923
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 1.0000 - loss: 0.6921
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step
Precisión Red Neuronal: 0.5


## Resumen y Comparación de Modelos

### 1. Modelo Naive Bayes (Scikit-learn)

*   **Enfoque:** Este modelo utiliza una tubería que primero vectoriza el texto usando TF-IDF (`TfidfVectorizer`) y luego aplica un clasificador Naive Bayes Multinomial (`MultinomialNB`). Naive Bayes se basa en el teorema de Bayes con la suposición de independencia entre las características (palabras).
*   **Resultados Obtenidos:**
    *   **Precisión:** 0.5
    *   **Reporte de Clasificación:**
        ```
                      precision    recall  f1-score   support

                 0       0.50      1.00      0.67         1
                 1       0.00      0.00      0.00         1

            accuracy                           0.50         2
           macro avg       0.25      0.50      0.33         2
        weighted avg       0.25      0.50      0.33         2
        ```
    *   El modelo clasificó correctamente 1 de 2 muestras del conjunto de prueba.

*   **Ventajas para este problema:**
    *   Sencillo y rápido de implementar y entrenar.
    *   Funciona bien con datos de alta dimensionalidad (como características TF-IDF).
    *   Requiere menos datos de entrenamiento en comparación con modelos más complejos.
*   **Desventajas para este problema:**
    *   La suposición de independencia de características rara vez se cumple en el lenguaje natural, lo que puede limitar su rendimiento.
    *   Su rendimiento puede ser inferior al de modelos más avanzados en conjuntos de datos grandes y complejos.

### 2. Modelo de Red Neuronal (TensorFlow/Keras LSTM)

*   **Enfoque:** Este modelo utiliza una red neuronal recurrente (LSTM) construida con TensorFlow y Keras. Primero tokeniza el texto y lo convierte en secuencias acolchadas. La red incluye una capa de Embedding para representar palabras en un espacio vectorial, seguida de una capa LSTM para capturar dependencias secuenciales, y capas Dense para la clasificación binaria.
*   **Resultados Obtenidos:**
    *   **Precisión Red Neuronal:** 0.5
    *   **Precisión en el entrenamiento (última época):** 1.0000
    *   El modelo también clasificó correctamente 1 de 2 muestras del conjunto de prueba.

*   **Ventajas para este problema:**
    *   Capacidad para aprender representaciones de palabras complejas y capturar relaciones contextuales (orden de las palabras).
    *   Potencialmente mayor rendimiento en tareas de PNL con conjuntos de datos grandes.
    *   Flexibilidad para construir arquitecturas más profundas y sofisticadas.
*   **Desventajas para este problema**
    *   Requiere una cantidad significativamente mayor de datos de entrenamiento para evitar el sobreajuste y generalizar bien.
    *   Mayor complejidad computacional y tiempo de entrenamiento.
    *   Más difícil de interpretar en comparación con modelos más simples.

### Comparación y Conclusión

Ambos modelos, Naive Bayes y la Red Neuronal LSTM, obtuvieron una **precisión de 0.5** en el conjunto de prueba. Esta baja precisión, sumada a las advertencias y la divergencia entre la precisión de entrenamiento (1.0) y prueba (0.5) en la red neuronal, indica que **ninguno de los modelos ha podido aprender y generalizar de manera efectiva** con el conjunto de datos de **solo 6 opiniones**.

**¿Cuál es mejor para este problema?**

Con un conjunto de datos tan extremadamente pequeño, es **difícil determinar cuál es mejor**. Ambos modelos tienen una carencia de datos.

*   **Naive Bayes:** Aunque simple, es más adecuado para conjuntos de datos pequeños si las suposiciones de independencia no son demasiado apartadas. Sin embargo, con solo 6 muestras, incluso este modelo lucha por encontrar patrones significativos.
*   **Red Neuronal (LSTM):** Los modelos de redes neuronales, especialmente los recurrentes como LSTM, requieren una gran cantidad de datos para entrenar sus múltiples parámetros y evitar el sobreajuste. El 100% de precisión en el entrenamiento y el 50% en la prueba en la red neuronal es un claro signo de **sobreajuste extremo** debido al tamaño insuficiente de los datos.

**Conclusión:**

Para el problema actual con este conjunto de datos minúsculo, **ningún modelo es adecuado**. Los resultados obtenidos no son representativos de su capacidad real.

Para una tarea de clasificación de sentimientos en un escenario real, si tuviéramos un conjunto de datos mucho más grande (miles o millones de opiniones):

*   Una **Red Neuronal (LSTM o Transformer)** tendría un **potencial significativamente mayor** para alcanzar una alta precisión, ya que puede capturar la complejidad y el contexto del lenguaje natural.
*   **Naive Bayes** podría servir como un **punto de partida rápido o una línea base** debido a su simplicidad y velocidad, pero probablemente sería superado por modelos más avanzados con suficientes datos.

Para mejorar los resultados, la prioridad fundamental sería **obtener un conjunto de datos de opiniones mucho más amplio y diverso**.